# Opalinuston as a host rock: a site-selection example

This notebook applies GemPy to the German search for a repository for high-level radioactive waste. We build a **schematic model of the Opalinuston (Opalinus Clay)**, one of the potential clay host rocks, and check it against some of the depth and thickness criteria that the BGE (Bundesgesellschaft für Endlagerung) applies in the current step of the site selection.

**Background: the Opalinuston sub-area (Teilgebiet)**

In the 2020 *Zwischenbericht Teilgebiete*, the BGE identified 90 sub-areas. One of them is the Teilgebiet *Mitteljura (Opalinuston)* (`001_00TG_032_01IG_T_f_jmOPT`) in eastern and southern Baden-Württemberg and western Bavaria:

- It covers **4,241 km²**, from the Swabian Alb into the northern Alpine foreland (Molasse Basin), mainly between Nördlingen, Reutlingen and Ravensburg, with a smaller part between Tuttlingen and Schaffhausen.
- The Opalinuston is **up to 160 m thick** and lies at **300–1,500 m depth**. It **deepens towards the southeast**.
- Most of the larger part sits on the *Süddeutsche Großscholle*, a block with few faults. Around the Danube it passes into the Molasse Basin, where faults become more frequent.
- In the north, the Opalinuston is overlain by the karstified Upper Jurassic (Malm), an important aquifer.

**What this notebook is, and what it is not**

The model below is **synthetic**. Its numbers (depths, thicknesses, dip, a fault) are chosen within the ranges reported by the BGE, but it **does not represent a real location** and the results are **not a site assessment**. The goal is to show how a 3D geological model can be used to evaluate geometric criteria, and how uncertainty in the model affects the result.

**Workflow:**
1. Synthetic borehole data for a 20 km × 10 km area across the transition from the Swabian Alb into the Molasse Basin
2. Structural model with GemPy: Molasse, Jurassic units and a normal fault
3. Map the Opalinuston: depth, thickness and distance to the fault
4. Apply site-selection criteria and compute the remaining area
5. Uncertainty: how stable is that area if the data are uncertain?

**Running in Google Colab?** Run the next cell first. It installs GemPy (takes 1-2 minutes).
If Colab asks you to restart the session afterwards, click *Restart* and continue with the cell after it.

Running locally (or on Binder)? The cell does nothing, so you can run it or skip it.

In [ ]:
# Install GemPy - only needed in Google Colab
import sys
if "google.colab" in sys.modules:
    %pip install -q "gempy[opt]==2026.0.3" "gempy_engine==2026.0.3.post1" "gempy_viewer[opt]==2026.0.5" "gempy_plugins==2026.0.3" "pyvista==0.49.2" "trame-pyvista==0.3.6" "nest_asyncio2==1.7.4" "anywidget==0.11.0"
else:
    print("Not running in Colab - nothing to install.")

In [ ]:
# Required imports
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gempy as gp
import gempy_viewer as gpv

In [ ]:
# 3D plot settings for Colab and Binder: show 3D plots directly in the notebook
# (locally nothing changes - you can skip this cell)
import os, sys
IN_COLAB = "google.colab" in sys.modules
IN_BINDER = "BINDER_LAUNCH_HOST" in os.environ or "BINDER_SERVICE_HOST" in os.environ
if IN_COLAB or IN_BINDER:
    import gempy_viewer as gpv

    PLOT_3D_BACKEND = 'html'  # interactive; use 'static' for plain images if 'html' doesn't work

    # gpv.plot_3d normally opens a separate window, which Colab and Binder can't show.
    # This wrapper makes it draw inside the notebook instead.
    _plot_3d = gpv.plot_3d
    def _plot_3d_colab(*args, show=True, image=False, kwargs_plotter=None, **kwargs):
        if image:
            return _plot_3d(*args, show=show, image=image, kwargs_plotter=kwargs_plotter, **kwargs)
        kwargs_plotter = {'notebook': True, **(kwargs_plotter or {})}
        gempy_vista = _plot_3d(*args, show=False, kwargs_plotter=kwargs_plotter, **kwargs)
        if show:
            gempy_vista.p.show(jupyter_backend=PLOT_3D_BACKEND)
        return gempy_vista
    gpv.plot_3d = _plot_3d_colab

### 1) Synthetic borehole data

We describe the model area with **local coordinates**: $x$ runs from the Swabian Alb in the northwest ($x = 0$) towards the Molasse Basin in the southeast ($x = 20$ km), $y$ along the strike of the layers. To keep depths simple, the ground surface is flat at 600 m above sea level, so

$$\text{depth below surface} = 600\ \text{m} - z.$$

The stratigraphy, from top to bottom (unit names as in the model):

| Unit | Description | Role |
|---|---|---|
| `Molasse` | Tertiary sediments of the Alpine foreland, thickening to the SE | overburden |
| `Oberjura` | Upper Jurassic (Malm) limestones, karstified | aquifer in the overburden |
| `Mitteljura` | Middle Jurassic above the Opalinuston | overburden |
| `Opalinuston` | Opalinuston Formation, clay rock | **host rock** |
| `Unterjura_Keuper` | Lower Jurassic and Keuper | underlying units |
| `basement` | Triassic and older (Muschelkalk, Buntsandstein, crystalline basement) | underlying units |

All Jurassic layers dip gently (about 2.5°) to the southeast. The Opalinuston thins from about 150 m in the NW to about 90 m in the SE. A normal fault strikes roughly along $y$ at $x \approx 7$ km, dips 70° to the SE and offsets the Jurassic by 80 m. It is older than the Molasse, so the Molasse is not offset.

In reality, such data come from boreholes, seismic profiles and existing 3D models of the state geological surveys. Here we "drill" 12 synthetic boreholes into a simple geometric model and record the depth of each formation boundary.

In [ ]:
GROUND_LEVEL = 600.0  # flat ground surface (m a.s.l.)

# Base of each unit as a plane z = a + b*x + c*y (footwall side of the fault)
# Gradients: about 2.5° dip to the ESE; the Opalinuston base dips less, so the layer thins to the SE
unit_bases = {
    "Oberjura":         (470.0, -0.045, -0.010),
    "Mitteljura":       (350.0, -0.045, -0.010),
    "Opalinuston":      (200.0, -0.042, -0.010),   # 150 m thick at x = 0, 90 m at x = 20 km
    "Unterjura_Keuper": (-100.0, -0.042, -0.010),
}
molasse_base = (650.0, -0.0425, 0.0)  # base of the Tertiary, crops out in the NW

# Normal fault: trace at z = 0 is x = 7000 + 0.2*(y - 5000), dipping 70° to the SE, 80 m throw
FAULT_DIP, FAULT_THROW = 70.0, 80.0

def fault_x(y, z):
    # x position of the fault plane at (y, z)
    return 7000.0 + 0.2 * (y - 5000.0) - z / np.tan(np.radians(FAULT_DIP)) * np.hypot(1, 0.2)

def plane_z(plane, x, y):
    a, b, c = plane
    return a + b * x + c * y

def plane_orientation(plane):
    # Azimuth (dip direction) and dip angle (degrees) of a plane z = a + b*x + c*y
    _, b, c = plane
    return np.degrees(np.arctan2(-b, -c)), np.degrees(np.arctan(np.hypot(b, c)))

In [ ]:
# 12 synthetic boreholes on two lines across the area
boreholes = pd.DataFrame(
    [(f"B{i + 1}", x, y) for i, (x, y) in enumerate(
        [(x, y) for y in (2500.0, 7500.0) for x in (1000.0, 4000.0, 9500.0, 12500.0, 16000.0, 19500.0)])],
    columns=["name", "X", "Y"],
)

rows = []
for _, bh in boreholes.iterrows():
    x, y = bh.X, bh.Y
    z_molasse = plane_z(molasse_base, x, y)
    if z_molasse < GROUND_LEVEL:  # no Molasse where its base would lie above ground
        rows.append((x, y, z_molasse, "Molasse"))
    for unit, plane in unit_bases.items():
        z = plane_z(plane, x, y)
        if x > fault_x(y, z):  # hanging wall: shifted down by the fault throw
            z -= FAULT_THROW
        rows.append((x, y, z, unit))

# The fault plane, e.g. mapped in seismic profiles
for y in (0.0, 5000.0, 10000.0):
    for z in (500.0, -500.0, -1300.0):
        rows.append((fault_x(y, z), y, z, "Fault"))

surface_points = pd.DataFrame(rows, columns=["X", "Y", "Z", "formation"]).round(1)

# Formation tops as depth below ground in the boreholes
picks = surface_points[surface_points.formation != "Fault"].copy()
picks["depth"] = GROUND_LEVEL - picks.Z
picks.pivot_table(index=["X", "Y"], columns="formation", values="depth", sort=False)[
    ["Molasse", "Oberjura", "Mitteljura", "Opalinuston", "Unterjura_Keuper"]].round(0)

The table shows the depth (m) of the **base** of each unit in the boreholes. In GemPy, the surface points of a unit mark its base, so the base of `Mitteljura` is the top of the Opalinuston. Empty cells mean the unit is not present.

Orientation data (dip and dip direction) are "measured" at every formation boundary in the boreholes, as a dipmeter log would provide them. Without them, GemPy has to extrapolate the dip towards the model edges from only 12 points.

In [ ]:
orientation_rows = []
for x, y, z, unit in surface_points[surface_points.formation != "Fault"].itertuples(index=False):
    plane = molasse_base if unit == "Molasse" else unit_bases[unit]
    azimuth, dip = plane_orientation(plane)
    orientation_rows.append((x, y, z, azimuth, dip, 1, unit))

# Fault orientation: dip direction perpendicular to the trace
fault_azimuth = np.degrees(np.arctan2(1.0, -0.2))
orientation_rows.append((fault_x(5000.0, -500.0), 5000.0, -500.0, fault_azimuth, FAULT_DIP, 1, "Fault"))

orientations = pd.DataFrame(orientation_rows,
                            columns=["X", "Y", "Z", "azimuth", "dip", "polarity", "formation"]).round(2)
orientations.head(8)

### 2) Structural model

We write the data to CSV files (the same format as in the other notebooks) and build the model. The structural frame has four groups, from young to old:

- `Tertiary`: the Molasse, which erodes the older units (base Tertiary unconformity)
- `Fault_Series`: the normal fault, which offsets the groups below it in the stack
- `Upper_Jurassic`: Oberjura and Mitteljura, i.e. everything down to the top of the Opalinuston
- `Lower_Jurassic`: the Opalinuston and the underlying units

The Jurassic is a conformable sequence, so why two groups? All surfaces in one group share one scalar field, and the orientation data push the surfaces to be parallel. A layer that thins, like the Opalinuston here, is then not reproduced well: in a single group, the modelled base of the Opalinuston misses the boreholes by up to about 30 m in the SE. With separate groups, the top and the base each get their own scalar field and honor the data.

In [ ]:
os.makedirs("opalinuston_data", exist_ok=True)
surface_points.to_csv("opalinuston_data/surface_points.csv", index=False)
orientations.to_csv("opalinuston_data/orientations.csv", index=False)

EXTENT = [0, 20000, 0, 10000, -1400, GROUND_LEVEL]
RESOLUTION = [80, 40, 160]  # cells of 250 m x 250 m x 12.5 m

def build_model(resolution=RESOLUTION, path="opalinuston_data/"):
    model = gp.create_geomodel(
        project_name="Opalinuston_example",
        extent=EXTENT,
        resolution=resolution,
        importer_helper=gp.data.ImporterHelper(
            path_to_orientations=path + "orientations.csv",
            path_to_surface_points=path + "surface_points.csv",
        ),
    )
    gp.map_stack_to_surfaces(
        gempy_model=model,
        mapping_object={
            "Tertiary": ("Molasse",),
            "Fault_Series": ("Fault",),
            "Upper_Jurassic": ("Oberjura", "Mitteljura"),
            "Lower_Jurassic": ("Opalinuston", "Unterjura_Keuper"),
        },
    )
    gp.set_is_fault(model, ["Fault_Series"])

    # Colors loosely following geological map conventions
    colors = {"Molasse": "#f2d16b", "Fault": "#222222", "Oberjura": "#7fb8e0",
              "Mitteljura": "#b07c4f", "Opalinuston": "#4b4b6b", "Unterjura_Keuper": "#c9a3c9"}
    for name, color in colors.items():
        model.structural_frame.get_element_by_name(name).color = color
    model.structural_frame.basement_color = "#d9d9d9"
    return model

geo_model = build_model()
geo_model.structural_frame

In [ ]:
gp.compute_model(geo_model)

In [ ]:
# Cross-section through the middle of the area (5x vertical exaggeration)
p2d = gpv.plot_2d(geo_model, direction="y", cell_number="mid", ve=5, show_data=False, show=False)
ax = p2d.axes[0]
for x in boreholes.X.unique():  # borehole positions, projected onto the section
    ax.axvline(x, color="k", lw=1, ls=":")
ax.set_title("Cross-section at y = 5 km (boreholes projected, 5x vertical exaggeration)")

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=element.color, edgecolor="k", linewidth=0.5, label=element.name)
                   for element in geo_model.structural_frame.structural_elements if element.name != "Fault"],
          loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
plt.show()

In [ ]:
# 3D view (5x vertical exaggeration)
gpv.plot_3d(geo_model, show_lith=False, ve=5)

### 3) Mapping the host rock

For each vertical column of the grid we find the top and base of the Opalinuston. We could count the Opalinuston cells in the lithology block, but then the precision is limited to the cell height (12.5 m). Instead we use GemPy's **scalar fields**: a surface lies where the scalar field of its group reaches the value of that surface, and interpolating between cell centres locates it much more precisely.

From top and base we derive:

- **top** and **base** of the Opalinuston and their depth below ground
- **thickness**
- the **repository level** (*Einlagerungsbereich*), here assumed in the middle of the layer
- the horizontal **distance to the fault** at the repository level. GemPy's *fault block* tells us on which side of the fault each cell lies, so the fault is where this value changes.

In [ ]:
def opalinuston_maps(model):
    nx, ny, nz = model.grid.dense_grid.resolution
    xyz = model.grid.dense_grid.values.reshape(nx, ny, nz, 3)
    x, y, z = xyz[:, :, 0, 0], xyz[:, :, 0, 1], xyz[0, 0, :, 2]  # z: cell centres, bottom to top
    dz = z[1] - z[0]
    raw = model.solutions.raw_arrays
    groups = [group.name for group in model.structural_frame.structural_groups]

    def surface_elevation(group, surface_index):
        # Elevation of the uppermost point in each column where the scalar field of `group`
        # reaches the value of its surface number `surface_index`
        i = groups.index(group)
        field = raw.scalar_field_matrix[i].reshape(nx, ny, nz)
        d = field - raw.scalar_field_at_surface_points[i][surface_index]
        crossing = np.sign(d[:, :, :-1]) != np.sign(d[:, :, 1:])
        k = nz - 2 - np.argmax(crossing[:, :, ::-1], axis=2)
        d0 = np.take_along_axis(d, k[..., None], axis=2)[..., 0]
        d1 = np.take_along_axis(d, k[..., None] + 1, axis=2)[..., 0]
        elevation = z[k] + dz * d0 / (d0 - d1)  # linear interpolation between cell centres
        return np.where(crossing.any(axis=2), elevation, np.nan)

    top = surface_elevation("Upper_Jurassic", 1)   # base of Mitteljura = top of Opalinuston
    base = surface_elevation("Lower_Jurassic", 0)  # base of Opalinuston
    present = np.isfinite(top) & np.isfinite(base) & (top > base)

    repository_z = (top + base) / 2
    maps = {
        "x": x, "y": y,
        "top_depth": GROUND_LEVEL - top,
        "thickness": np.where(present, top - base, 0.0),
        "repository_depth": GROUND_LEVEL - repository_z,
    }

    # Horizontal distance to the fault at the repository level.
    # fault_block: side of the fault for every cell; the fault lies where it changes.
    fault_side = model.solutions.raw_arrays.fault_block.reshape(nx, ny, nz)
    level = np.clip(np.round((np.nan_to_num(repository_z) - z[0]) / dz).astype(int), 0, nz - 1)
    distance = np.full((nx, ny), np.nan)
    for k in np.unique(level[present]):
        side = fault_side[:, :, k]
        at_fault = np.zeros_like(side, dtype=bool)  # cells next to a cell on the other side
        at_fault[:-1] |= side[:-1] != side[1:]
        at_fault[1:] |= side[1:] != side[:-1]
        at_fault[:, :-1] |= side[:, :-1] != side[:, 1:]
        at_fault[:, 1:] |= side[:, 1:] != side[:, :-1]
        columns = (level == k) & present
        if not at_fault.any():  # fault not present at this depth
            distance[columns] = np.inf
            continue
        fault_xy = np.stack([x[at_fault], y[at_fault]], axis=1)
        column_xy = np.stack([x[columns], y[columns]], axis=1)
        distance[columns] = np.linalg.norm(column_xy[:, None] - fault_xy[None], axis=-1).min(axis=1)
    maps["fault_distance"] = distance
    return maps

maps = opalinuston_maps(geo_model)
print(f"Opalinuston top:       {np.nanmin(maps['top_depth']):.0f} - {np.nanmax(maps['top_depth']):.0f} m below ground")
print(f"Opalinuston thickness: {np.nanmin(maps['thickness']):.0f} - {np.nanmax(maps['thickness']):.0f} m")

In [ ]:
map_extent = EXTENT[:4]
km = lambda v: v / 1000

def show_map(ax, values, title, cmap="viridis", label="", **kwargs):
    im = ax.imshow(values.T, origin="lower", extent=[km(v) for v in map_extent], cmap=cmap,
                   aspect="equal", **kwargs)
    ax.set_title(title)
    ax.set_xlabel("x (km), NW → SE")
    ax.set_ylabel("y (km)")
    plt.colorbar(im, ax=ax, label=label, shrink=0.8)
    return im

fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
show_map(axes[0, 0], maps["top_depth"], "Depth of the Opalinuston top", "viridis_r", "m below ground")
show_map(axes[0, 1], maps["thickness"], "Thickness of the Opalinuston", "cividis", "m")
show_map(axes[1, 0], maps["repository_depth"], "Depth of the repository level", "viridis_r", "m below ground")
show_map(axes[1, 1], maps["fault_distance"], "Distance to the fault", "magma", "m", vmax=3000)
plt.show()

Note the thin strip in the thickness map along the fault. Where a vertical column crosses the normal fault inside the Opalinuston, part of the layer is cut out, so a borehole there would find it thinner than it really is (a *missing section*). Faults therefore also complicate the characterization of the host rock.

### 4) Applying site-selection criteria

We now check every column against four geometric criteria. The first three are taken from the legal and BGE requirements for clay rock; the fourth is an **assumption for this exercise**:

| Criterion | Requirement | Source |
|---|---|---|
| Thickness | Opalinuston at least **100 m** thick | minimum requirement, § 23 StandAG |
| Top depth | top of the host rock at least **300 m** below ground | minimum requirement, § 23 StandAG |
| Repository depth | repository level **not deeper than 1,000 m**; deeper is rated "unfavourable" | rvSU criterion "Ungünstige Tiefenlage des Endlagerbergwerks" (BGE, Forum Endlagersuche 2024) |
| Fault distance | at least **500 m** from the fault | *assumption for this example*, not a BGE value |

The BGE evaluates many more criteria, for example barrier thickness, lateral extent, spatial predictability, the tectonic situation, glacial erosion channels, and the overburden. In the real procedure these are weighed against each other in several steps (Prüfschritte 1–4). Here we only look at the geometric part.

In [ ]:
MIN_THICKNESS = 100.0       # m, § 23 StandAG
MIN_TOP_DEPTH = 300.0       # m below ground, § 23 StandAG
MAX_REPOSITORY_DEPTH = 1000.0  # m below ground, rvSU criterion
MIN_FAULT_DISTANCE = 500.0  # m, assumption for this example

def apply_criteria(maps):
    return {
        f"Thickness ≥ {MIN_THICKNESS:.0f} m": maps["thickness"] >= MIN_THICKNESS,
        f"Top ≥ {MIN_TOP_DEPTH:.0f} m below ground": maps["top_depth"] >= MIN_TOP_DEPTH,
        f"Repository level ≤ {MAX_REPOSITORY_DEPTH:.0f} m": maps["repository_depth"] <= MAX_REPOSITORY_DEPTH,
        f"Distance to fault ≥ {MIN_FAULT_DISTANCE:.0f} m": maps["fault_distance"] >= MIN_FAULT_DISTANCE,
    }

criteria = apply_criteria(maps)
all_met = np.logical_and.reduce(list(criteria.values()))

cell_area_km2 = (EXTENT[1] - EXTENT[0]) / RESOLUTION[0] * (EXTENT[3] - EXTENT[2]) / RESOLUTION[1] / 1e6
for name, ok in criteria.items():
    print(f"{name:32s} met on {ok.sum() * cell_area_km2:6.1f} km²")
print(f"{'All criteria':32s} met on {all_met.sum() * cell_area_km2:6.1f} km² "
      f"of {all_met.size * cell_area_km2:.0f} km²")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5), constrained_layout=True)
for ax, (name, ok) in zip(axes.flat, criteria.items()):
    show_map(ax, ok.astype(float), name, "RdYlGn", vmin=0, vmax=1)

# Number of failed criteria per column
n_failed = sum((~ok).astype(int) for ok in criteria.values())
show_map(axes.flat[4], n_failed, "Number of failed criteria", "Reds", vmin=0, vmax=len(criteria))
show_map(axes.flat[5], all_met.astype(float), f"All criteria met: {all_met.sum() * cell_area_km2:.0f} km²",
         "RdYlGn", vmin=0, vmax=1)
plt.show()

Each criterion removes a different part of the model:

- In the **NW corner**, the top of the Opalinuston comes closer than 300 m to the surface.
- A strip about **1 km wide along the fault** is excluded by the distance criterion. It splits the remaining area into two blocks.
- Towards the **SE**, the repository level gets deeper than 1,000 m, and beyond about 17 km the layer is also thinner than 100 m, so two criteria fail there.

The BGE's 2020 report for this Teilgebiet uses an area of **10 km²** for the containment zone (*einschlusswirksamer Gebirgsbereich*) in clay rock as a reference. In this synthetic example, the remaining area is clearly larger. But how reliable is it, given that the model is based on only 12 boreholes?

### 5) Uncertainty: how stable is the suitable area?

The borehole depths are not exact, and between the boreholes the geometry is interpolated. As in the *Probabilistic_modeling* notebook, we draw random variations of the input data and recompute the model many times (Monte Carlo):

- the depth of the Jurassic units at each borehole shifts by a random amount (standard deviation 40 m, the same shift for all units in one borehole)
- the position of the fault shifts in $x$ (standard deviation 300 m)

For each realization we apply the criteria again. The result is a **probability map**: in how many realizations does a column meet all criteria?

To keep this fast, the realizations use a coarser grid and skip the extraction of surface meshes, which we do not need here. 25 realizations take about two minutes.

In [ ]:
import contextlib
import io

rng = np.random.default_rng(42)
N_REALIZATIONS = 25
MC_RESOLUTION = [40, 10, 100]  # cells of 500 m x 1000 m x 20 m
os.makedirs("opalinuston_data/mc", exist_ok=True)

jurassic_units = list(unit_bases)
probability = np.zeros(MC_RESOLUTION[:2])
suitable_area = []

for n in range(N_REALIZATIONS):
    print(f"Realization {n + 1} of {N_REALIZATIONS}", end="\r")
    sp = surface_points.copy()
    ori = orientations.copy()

    # Correlated depth error per borehole for all Jurassic units
    for (x, y), idx in sp[sp.formation.isin(jurassic_units)].groupby(["X", "Y"]).groups.items():
        sp.loc[idx, "Z"] += rng.normal(0, 40)

    # Shift the fault in x
    fault_shift = rng.normal(0, 300)
    sp.loc[sp.formation == "Fault", "X"] += fault_shift
    ori.loc[ori.formation == "Fault", "X"] += fault_shift

    sp.to_csv("opalinuston_data/mc/surface_points.csv", index=False)
    ori.to_csv("opalinuston_data/mc/orientations.csv", index=False)

    with contextlib.redirect_stdout(io.StringIO()):  # hide GemPy's progress messages
        model = build_model(resolution=MC_RESOLUTION, path="opalinuston_data/mc/")
        model.interpolation_options.mesh_extraction = False
        gp.compute_model(model)

    ok = np.logical_and.reduce(list(apply_criteria(opalinuston_maps(model)).values()))
    probability += ok / N_REALIZATIONS
    mc_cell_area = cell_area_km2 * (RESOLUTION[0] / MC_RESOLUTION[0]) * (RESOLUTION[1] / MC_RESOLUTION[1])
    suitable_area.append(ok.sum() * mc_cell_area)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True,
                         gridspec_kw={"width_ratios": [2, 1]})
show_map(axes[0], probability, "Probability that all criteria are met", "RdYlGn", "probability",
         vmin=0, vmax=1)
axes[1].hist(suitable_area, bins=10, color="steelblue", edgecolor="white")
axes[1].axvline(all_met.sum() * cell_area_km2, color="k", ls="--", label="original model")
axes[1].set_xlabel("Area meeting all criteria (km²)")
axes[1].set_ylabel("Number of realizations")
axes[1].legend()
plt.show()

print(f"Area meeting all criteria: {np.mean(suitable_area):.0f} ± {np.std(suitable_area):.0f} km²")
print(f"Area with probability ≥ 0.9: {(probability >= 0.9).sum() * mc_cell_area:.0f} km²")

The core of the two blocks is stable, but their edges move: in the NW corner, along the fault, and especially at the SE limit where the repository level reaches 1,000 m depth. Because the layers dip only about 2.5°, an error of 40 m in depth shifts the 1,000 m line by roughly 1 km horizontally. Small vertical uncertainties can therefore have a large effect on the area that remains.

This is one reason why the site-selection process continues with surface exploration (seismic surveys, boreholes) in the siting regions, and why uncertainty is part of the safety assessment.

### Exercises

1. **Fault distance:** the 500 m distance is an assumption. How does the remaining area change for 250 m or 1,000 m?
2. **Thinner host rock:** raise the base of the Opalinuston by 30 m (first value of `"Opalinuston"` in `unit_bases`: 200 → 230), so that it is 120 m thick in the NW. Where does the thickness criterion start to matter?
3. **Larger offset:** increase `FAULT_THROW` to 200 m. The BGE describes offsets of 200–250 m for faults in the Hegau–Lake Constance region, where it rated the tectonic situation unfavourable. What happens to the Opalinuston at the fault?
4. **More data:** add boreholes, or reduce the depth uncertainty in the Monte Carlo loop. How does the probability map change?

### Outlook

- The model can be passed on to process simulations, e.g. heat transport from the waste canisters (the BGE uses design temperatures for each host rock), groundwater flow in the overburden, or geomechanics.
- Real 3D models of the area exist at the state geological surveys (e.g. LGRB Baden-Württemberg, LfU Bayern) and in the BGE's own models used for the rvSU.
- In Switzerland, Nagra proposed the siting region *Nördlich Lägern* in the Opalinus Clay in 2022; the same formation is studied in the Mont Terri rock laboratory.

### Sources

- BGE (2025): [Geologischer Steckbrief – Teilgebiet Mitteljura (Opalinuston)](https://www.bge.de/fileadmin/user_upload/Standortsuche/Wesentliche_Unterlagen/BGE_Navigator/Geologische_Steckbriefe/20241104_Geologischer_Steckbrief_-_Teilgebiet_001_00TG_032_01IG_T_f_jmOPT_barrierefrei.pdf), status 03.11.2025
- BGE (2026): [rvSU-Arbeitsstand 2026 – Teilgebiet Mitteljura (Opalinuston)](https://www.bge.de/fileadmin/user_upload/Standortsuche/Wesentliche_Unterlagen/BGE_Navigator/rvSU-Arbeitsst%C3%A4nde/2026/20261027_rvSU-Arbeitsstand_2026_-_Teilgebiet_Mitteljura-Opalinuston_barrierefrei.pdf), status 27.10.2026
- BGE (2020): [Ergebnisse der Bewertung, Teilgebiet 001_00TG_032_01IG_T_f_jmOPT](https://www.bge.de/fileadmin/user_upload/Standortsuche/Wesentliche_Unterlagen/Zwischenbericht_Teilgebiete/Lang-Steckbriefe/001_00TG_032_01IG_T_f_jmOPT.pdf), Zwischenbericht Teilgebiete
- BGE (2024): [Einengung im Rahmen der rvSU im Wirtsgestein Tongestein](https://www.bge.de/fileadmin/user_upload/Standortsuche/Wesentliche_Unterlagen/07_-_Vortraege/Vortraege_Schritt_2_Phase_1/Forum_Endlagersuche/Forum_2024/20241122_Pr%C3%A4sentation_Einengung_im_Rahmen_der_rvSU_im_Wirtsgestein_Tongestein.pdf), 3. Forum Endlagersuche, 22.11.2024
- [BGE Endlagersuche Navigator](https://www.bge.de/de/endlagersuche/) (interactive map of the current Arbeitsstand)
- Standortauswahlgesetz (StandAG), § 23 Mindestanforderungen